In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error  # 模型评估指标
from tensorflow.keras.layers import LSTM, Dense, Dropout  # 增加Dropout防止过拟合
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping  # 早停机制，避免训练过度

# ------------------------------------------------------------------------------
# 1. 数据加载与初步探索（以CSV格式的股价数据为例，需替换为你的数据路径）
# ------------------------------------------------------------------------------
def load_data(file_path):
    """加载数据并提取收盘价（时序预测核心特征）"""
    # 读取CSV数据（假设数据包含"date"日期列和"close"收盘价列）
    df = pd.read_csv(file_path)
    
    # 数据校验：检查关键列是否存在
    required_cols = ["date", "close"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError(f"数据缺少必要列！需包含：{required_cols}")
    
    # 转换日期格式并排序（时序数据必须按时间顺序排列）
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)  # 按时间升序排列
    
    # 提取收盘价作为预测目标（LSTM输入需为二维数据：[样本数, 特征数]）
    data = df[["close"]].values  # shape: (n_samples, 1)
    return df, data

# 替换为你的数据路径（如本地CSV、Excel，或通过API获取）
df, data = load_data(file_path="stock_data.csv")
print(f"数据总量：{len(df)} 条，时间范围：{df['date'].min()} ~ {df['date'].max()}")


# ------------------------------------------------------------------------------
# 2. 数据预处理（时序预测核心步骤：归一化+构建监督学习样本）
# ------------------------------------------------------------------------------
def preprocess_data(data, time_step=60, test_size=0.2):
    """
    时序数据预处理：
    1. 归一化（LSTM对数值范围敏感，需缩放到[0,1]）
    2. 构建监督学习样本（用前time_step天数据预测第time_step+1天收盘价）
    """
    # 1. 归一化（仅用训练集拟合scaler，避免数据泄露）
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(data)  # shape: (n_samples, 1)
    
    # 2. 划分训练集/测试集（按时间顺序划分，避免打乱时序）
    train_size = int(len(data_scaled) * (1 - test_size))
    train_data = data_scaled[:train_size, :]
    test_data = data_scaled[train_size:, :]
    
    # 3. 构建监督学习样本（X: 前time_step天数据，y: 第time_step+1天数据）
    def create_samples(dataset, time_step):
        X, y = [], []
        for i in range(len(dataset) - time_step - 1):
            # 取前time_step个数据作为输入特征X
            X.append(dataset[i:(i + time_step), 0])
            # 取第time_step+1个数据作为标签y
            y.append(dataset[i + time_step, 0])
        return np.array(X), np.array(y)
    
    # 生成训练集/测试集的X和y
    X_train, y_train = create_samples(train_data, time_step)
    X_test, y_test = create_samples(test_data, time_step)
    
    # 4. 调整LSTM输入格式：[样本数, 时间步长, 特征数]（LSTM要求3D输入）
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    
    print(f"训练集：X_train.shape={X_train.shape}, y_train.shape={y_train.shape}")
    print(f"测试集：X_test.shape={X_test.shape}, y_test.shape={y_test.shape}")
    return X_train, y_train, X_test, y_test, scaler

# 超参数：时间步长（用前60天预测第61天），测试集占比20%
X_train, y_train, X_test, y_test, scaler = preprocess_data(data, time_step=60, test_size=0.2)


# ------------------------------------------------------------------------------
# 3. 搭建并训练LSTM模型（优化结构：避免过拟合+稳定训练）
# ------------------------------------------------------------------------------
def build_lstm_model(input_shape):
    """搭建LSTM模型：含Dropout防止过拟合，Dense输出预测值"""
    model = Sequential()
    
    # 第一层LSTM：64个神经元，返回序列（供下一层LSTM使用），输入形状=（时间步长，特征数）
    model.add(LSTM(units=64, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))  # Dropout=0.2：随机丢弃20%神经元，防止过拟合
    
    # 第二层LSTM：32个神经元，不返回序列（最后一层LSTM）
    model.add(LSTM(units=32, return_sequences=False))
    model.add(Dropout(0.2))
    
    # 全连接层：输出1个预测值（收盘价）
    model.add(Dense(units=1))
    
    # 编译模型：优化器用Adam（常用且稳定），损失函数用MSE（回归任务标准）
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model

# 输入形状：（时间步长，特征数）=（X_train.shape[1], X_train.shape[2]）
input_shape = (X_train.shape[1], X_train.shape[2])
model = build_lstm_model(input_shape)
model.summary()  # 打印模型结构（查看层数、参数数量）

# 训练模型：早停机制（val_loss连续5轮不下降则停止，避免过拟合）
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    batch_size=32,  # 批次大小（根据数据量调整，32/64常用）
    epochs=50,      # 最大训练轮次
    validation_data=(X_test, y_test),  # 用测试集验证损失
    callbacks=[early_stop],  # 加入早停机制
    verbose=1       # 显示训练过程（1=显示进度条，0=不显示）
)


# ------------------------------------------------------------------------------
# 4. 模型预测与结果评估（反归一化+计算误差+可视化）
# ------------------------------------------------------------------------------
def evaluate_model(model, X_train, y_train, X_test, y_test, scaler):
    """模型评估：计算MSE/MAE误差，反归一化还原真实值"""
    # 1. 预测训练集和测试集
    train_predict = model.predict(X_train)
    test_predict = model.predict(X_test)
    
    # 2. 反归一化（将[0,1]的预测值还原为真实收盘价）
    train_predict = scaler.inverse_transform(train_predict)
    test_predict = scaler.inverse_transform(test_predict)
    y_train_true = scaler.inverse_transform(y_train.reshape(-1, 1))  # y需转为二维再反归一化
    y_test_true = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    # 3. 计算评估指标（MSE：均方误差，MAE：平均绝对误差，值越小模型越好）
    train_mse = mean_squared_error(y_train_true, train_predict)
    test_mse = mean_squared_error(y_test_true, test_predict)
    train_mae = mean_absolute_error(y_train_true, train_predict)
    test_mae = mean_absolute_error(y_test_true, test_predict)
    
    print("\n=== 模型评估结果 ===")
    print(f"训练集 MSE：{train_mse:.4f}，MAE：{train_mae:.4f}")
    print(f"测试集 MSE：{test_mse:.4f}，MAE：{test_mae:.4f}")
    return train_predict, test_predict, y_train_true, y_test_true

# 评估模型并获取预测结果
train_predict, test_predict, y_train_true, y_test_true = evaluate_model(
    model, X_train, y_train, X_test, y_test, scaler
)


# ------------------------------------------------------------------------------
# 5. 可视化结果（训练损失曲线+真实值vs预测值对比）
# ------------------------------------------------------------------------------
def plot_results(df, train_predict, test_predict, time_step):
    """可视化：1. 训练损失曲线；2. 真实收盘价与预测值对比"""
    # 设置中文显示
    plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
    plt.rcParams['axes.unicode_minus'] = False
    
    # 创建2个子图（1行2列）
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 子图1：训练损失曲线（查看是否过拟合：val_loss上升则过拟合）
    ax1.plot(history.history["loss"], label="训练损失", color="#1f77b4")
    ax1.plot(history.history["val_loss"], label="验证损失", color="#ff7f0e")
    ax1.set_title("LSTM模型训练损失曲线", fontsize=13, pad=15)
    ax1.set_xlabel("训练轮次（Epochs）", fontsize=11)
    ax1.set_ylabel("均方误差（MSE）", fontsize=11)
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 子图2：真实收盘价与预测值对比（时序顺序）
    # 构建完整的预测序列（训练集预测+测试集预测，对齐原始数据索引）
    train_predict_plot = np.empty_like(data)  # 与原始数据同形状的空数组
    train_predict_plot[:, :] = np.nan  # 填充NaN（未预测部分不显示）
    # 训练集预测值的索引：从time_step开始（前time_step天无预测）
    train_predict_plot[time_step:len(train_predict) + time_step, :] = train_predict
    
    test_predict_plot = np.empty_like(data)
    test_predict_plot[:, :] = np.nan
    # 测试集预测值的索引：从训练集结束位置+time_step开始
    test_start_idx = len(train_predict) + time_step * 2 + 1
    test_predict_plot[test_start_idx:len(data) - 1, :] = test_predict
    
    # 绘制原始数据、训练集预测、测试集预测
    ax2.plot(scaler.inverse_transform(data), label="真实收盘价", color="#2ca02c", linewidth=1.5)
    ax2.plot(train_predict_plot, label="训练集预测值", color="#1f77b4", linestyle="--", linewidth=1.2)
    ax2.plot(test_predict_plot, label="测试集预测值", color="#ff7f0e", linestyle="--", linewidth=1.2)
    ax2.set_title("LSTM收盘价预测结果对比", fontsize=13, pad=15)
    ax2.set_xlabel("时间序列（数据点）", fontsize=11)
    ax2.set_ylabel("收盘价（元）", fontsize=11)
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 调整布局并保存图片
    plt.tight_layout()
    plt.savefig("lstm_stock_prediction.png", dpi=300, bbox_inches="tight")
    plt.show()

# 调用可视化函数（time_step与预处理时一致）
plot_results(df, train_predict, test_predict, time_step=60)


# ------------------------------------------------------------------------------
# 6. 模型保存（可选：保存训练好的模型，后续直接加载使用）
# ------------------------------------------------------------------------------
model.save("lstm_stock_model.h5")  # 保存为H5格式（兼容TensorFlow/Keras）
print("\n模型已保存为：lstm_stock_model.h5")

FileNotFoundError: [Errno 2] No such file or directory: 'stock_data.csv'